In this build, we are building a xgboost model. 

The imputation methods are:
- Categorical: fill in with "NA"
- Numeric: fill in with -999

In [ ]:
import pandas as pd
import scipy as sp
from sklearn import preprocessing
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn import cross_validation
import xgboost as xgb
import pprint as pp
from xgboost.sklearn import XGBClassifier
from sklearn import cross_validation, metrics   #Additional scklearn functions
from sklearn.grid_search import GridSearchCV   #Perforing grid search
import bnp_helper
import common_helper
import random
import os
import datetime
from sklearn.externals import joblib
from matplotlib.pylab import rcParams

%matplotlib inline
rcParams['figure.figsize'] = 16, 4
pd.set_option('display.max_columns', 500)

In [ ]:
################### Combine the training set and test set to fix the unmatched categories ############
train_df_raw = pd.read_csv("../data/train.csv")
test_df_raw = pd.read_csv("../data/test.csv")

df_raw_combined = pd.concat([train_df_raw, test_df_raw], axis = 0)

########### Clean and Impute (Combined) #############
df_combinded = bnp_helper.clean(df_raw_combined, drop_collinearity = True, inplace = True)
df_combinded = bnp_helper.impute_cate_with_na_numeric_with_outlier(df_combinded)

################ Convert text to number (Combined) ################
df_combinded = common_helper.dummify(df_combinded, bnp_helper.get_categorical_variables(df_combinded))
print df_combinded.shape
df_combinded.head()

In [ ]:
train_df = df_combinded[-df_raw_combined['target'].isnull()]
test_df = df_combinded[df_raw_combined['target'].isnull()]

train_df_sample = train_df.sample(40000, random_state = 0)
#train_df_sample = train_df
target_train = train_df_sample['target']
X_train = train_df_sample.drop(['ID', 'target'], axis = 1)

In [ ]:
######### Set initial values for the model ######
xgb_model = XGBClassifier(
    learning_rate =0.1,
    n_estimators=1000,
    max_depth=5,
    min_child_weight=1,
    gamma=0,
    subsample=0.8,
    colsample_bytree=0.8,
    objective= 'binary:logistic',
    nthread=8,
    scale_pos_weight=1,
    seed=27)

start = datetime.datetime.now()
bnp_helper.xgboost_model_fit(xgb_model, X_train, target_train, ['logloss'])
end = datetime.datetime.now()

print "model run time: {}".format(end - start)

bnp_helper.feature_importance_plot(xgb_model.booster().get_fscore(), 100)

In [ ]:
################# Tune max_depth and min_child_weight ##############
param_test1 = {
    'max_depth':range(3,10,2),
    'min_child_weight':range(1,6,2)
}

gsearch1 = GridSearchCV(estimator = XGBClassifier(learning_rate =0.1, n_estimators=100, max_depth=5, \
    min_child_weight=3, gamma=0.3, subsample=0.8, colsample_bytree=0.8, \
    objective= 'binary:logistic', nthread=8, scale_pos_weight=1, seed=27), \
    param_grid = param_test1, scoring='log_loss',n_jobs=4,iid=False, cv=5)

start = datetime.datetime.now()
gsearch1.fit(X_train, target_train)
end = datetime.datetime.now()
print "model training time: {}".format(end - start)

gsearch1.grid_scores_, gsearch1.best_params_, gsearch1.best_score_

In [ ]:
param_test2 = {
    'max_depth':[4,5,6],
    'min_child_weight':[4,5,6]
}

gsearch1 = GridSearchCV(estimator = XGBClassifier(learning_rate =0.1, n_estimators=100, max_depth=5, \
    min_child_weight=3, gamma=0.3, subsample=0.8, colsample_bytree=0.8, \
    objective= 'binary:logistic', nthread=4, scale_pos_weight=1, seed=27), \
    param_grid = param_test2, scoring='log_loss',n_jobs=4,iid=False, cv=5)

start = datetime.datetime.now()
gsearch1.fit(X_train, target_train)
end = datetime.datetime.now()
print "model training time: {}".format(end - start)

gsearch1.grid_scores_, gsearch1.best_params_, gsearch1.best_score_

In [ ]:
param_test3 = {
    'max_depth':[6,7],
    'min_child_weight':[6,7,8]
}

gsearch1 = GridSearchCV(estimator = XGBClassifier(learning_rate =0.1, n_estimators=100, max_depth=5, \
    min_child_weight=3, gamma=0.3, subsample=0.8, colsample_bytree=0.8, \
    objective= 'binary:logistic', nthread=4, scale_pos_weight=1, seed=27), \
    param_grid = param_test3, scoring='log_loss',n_jobs=4,iid=False, cv=5)

start = datetime.datetime.now()
gsearch1.fit(X_train, target_train)
end = datetime.datetime.now()
print "model training time: {}".format(end - start)

gsearch1.grid_scores_, gsearch1.best_params_, gsearch1.best_score_

In [ ]:
################### tune gamma ################
param_test4 = {
    'max_depth':[6],
    'min_child_weight':[7], 
    'gamma':[i/10.0 for i in range(0,10,2)]
}

gsearch1 = GridSearchCV(estimator = XGBClassifier(learning_rate =0.1, n_estimators=100, max_depth=5, \
    min_child_weight=3, gamma=0.3, subsample=0.8, colsample_bytree=0.8, \
    objective= 'binary:logistic', nthread=4, scale_pos_weight=1, seed=27), \
    param_grid = param_test4, scoring='log_loss',n_jobs=4,iid=False, cv=5)

start = datetime.datetime.now()
gsearch1.fit(X_train, target_train)
end = datetime.datetime.now()
print "model training time: {}".format(end - start)

gsearch1.grid_scores_, gsearch1.best_params_, gsearch1.best_score_

In [ ]:
param_test5 = {
    'max_depth':[6],
    'min_child_weight':[7], 
    'gamma':[0.3, 0.4, 0.5]
}

gsearch1 = GridSearchCV(estimator = XGBClassifier(learning_rate =0.1, n_estimators=100, max_depth=5, \
    min_child_weight=3, gamma=0.3, subsample=0.8, colsample_bytree=0.8, \
    objective= 'binary:logistic', nthread=4, scale_pos_weight=1, seed=27), \
    param_grid = param_test5, scoring='log_loss',n_jobs=4,iid=False, cv=5)

start = datetime.datetime.now()
gsearch1.fit(X_train, target_train)
end = datetime.datetime.now()
print "model training time: {}".format(end - start)

gsearch1.grid_scores_, gsearch1.best_params_, gsearch1.best_score_

In [ ]:
################ Tune subsample and colsample_bytree ###############
param_test6 = {
    'max_depth':[6],
    'min_child_weight':[7], 
    'gamma':[0.4],
    'subsample':[i/10.0 for i in range(6,10)],
    'colsample_bytree':[i/10.0 for i in range(6,10)]
}

gsearch1 = GridSearchCV(estimator = XGBClassifier(learning_rate =0.1, n_estimators=100, max_depth=5, \
    min_child_weight=3, gamma=0.3, subsample=0.8, colsample_bytree=0.8, \
    objective= 'binary:logistic', nthread=4, scale_pos_weight=1, seed=27), \
    param_grid = param_test6, scoring='log_loss',n_jobs=4,iid=False, cv=5)

start = datetime.datetime.now()
gsearch1.fit(X_train, target_train)
end = datetime.datetime.now()
print "model training time: {}".format(end - start)

gsearch1.grid_scores_, gsearch1.best_params_, gsearch1.best_score_

In [ ]:
param_test7 = {
    'max_depth':[6],
    'min_child_weight':[7], 
    'gamma':[0.4],
    'subsample':[0.75, 0.8, 0.85],
    'colsample_bytree':[0.75, 0.8, 0.85]
}

gsearch1 = GridSearchCV(estimator = XGBClassifier(learning_rate =0.1, n_estimators=100, max_depth=5, \
    min_child_weight=3, gamma=0.3, subsample=0.8, colsample_bytree=0.8, \
    objective= 'binary:logistic', nthread=4, scale_pos_weight=1, seed=27), \
    param_grid = param_test7, scoring='log_loss',n_jobs=4,iid=False, cv=5)

start = datetime.datetime.now()
gsearch1.fit(X_train, target_train)
end = datetime.datetime.now()
print "model training time: {}".format(end - start)

gsearch1.grid_scores_, gsearch1.best_params_, gsearch1.best_score_

In [ ]:
################ Tune reg_alpha #############
param_test8 = {
    'max_depth':[6],
    'min_child_weight':[7], 
    'gamma':[0.4],
    'subsample':[0.85],
    'colsample_bytree':[0.85],
    'reg_alpha':[1e-5, 1e-2, 0.1, 1, 100]
}

gsearch1 = GridSearchCV(estimator = XGBClassifier(learning_rate =0.1, n_estimators=100, max_depth=5, \
    min_child_weight=3, gamma=0.3, subsample=0.8, colsample_bytree=0.8, \
    objective= 'binary:logistic', nthread=4, scale_pos_weight=1, seed=27), \
    param_grid = param_test8, scoring='log_loss',n_jobs=4,iid=False, cv=5)

start = datetime.datetime.now()
gsearch1.fit(X_train, target_train)
end = datetime.datetime.now()
print "model training time: {}".format(end - start)

gsearch1.grid_scores_, gsearch1.best_params_, gsearch1.best_score_

In [ ]:
param_test9 = {
    'max_depth':[6],
    'min_child_weight':[7], 
    'gamma':[0.4],
    'subsample':[0.85],
    'colsample_bytree':[0.85],
    'reg_alpha':[1e-6, 1e-5, 1e-4]
}

gsearch1 = GridSearchCV(estimator = XGBClassifier(learning_rate =0.1, n_estimators=100, max_depth=5, \
    min_child_weight=3, gamma=0.3, subsample=0.8, colsample_bytree=0.8, \
    objective= 'binary:logistic', nthread=4, scale_pos_weight=1, seed=27), \
    param_grid = param_test9, scoring='log_loss',n_jobs=4,iid=False, cv=5)

start = datetime.datetime.now()
gsearch1.fit(X_train, target_train)
end = datetime.datetime.now()
print "model training time: {}".format(end - start)

gsearch1.grid_scores_, gsearch1.best_params_, gsearch1.best_score_

In [ ]:
#train_df_sample = train_df.sample(10000, random_state = 0)
train_df_sample = train_df
target_train = train_df_sample['target']
X_train = train_df_sample.drop(['ID', 'target'], axis = 1)

In [ ]:
# Fit the model with best parameters and reduced learing rate
xgb_model = XGBClassifier(
    learning_rate =0.05,    # learning rate
    n_estimators=1000,
    max_depth=6,           # depth of tree, typical value 3-10, tunned
    min_child_weight=7,    # min number of observations in a node, tunned
    gamma=0.4,             # min loss reduction to make a split, tunned
    subsample=0.85,         # number of rows, tunned
    colsample_bytree=0.85,  # number of columns, tunned
    reg_alpha=0.0001,       # L1 regularization term on weight (analogous to Lasso regression)
    objective= 'binary:logistic',
    nthread=6,
    scale_pos_weight=1,    # > 0 value in case of high class imbalance
    seed=27)

In [ ]:
###### Fit the best model with training data #######
start = datetime.datetime.now()
bnp_helper.xgboost_model_fit(xgb_model, X_train, target_train, ["logloss"])
end = datetime.datetime.now()

print "model training time: {}".format(end - start)

bnp_helper.feature_importance_plot(xgb_model.booster().get_fscore(), 100)


In [ ]:
joblib.dump(xgb_model, 'xgb_model_2.pkl')

In [ ]:
################# Predict the test set and save result to file ############
#test_df_sample = test_df.sample(1000, random_state = 0)
test_df_sample = test_df

X_test = test_df_sample.drop(['ID', 'target'], axis = 1)

#Predict test set:
predprob_test = xgb_model.predict_proba(X_test)
        
# Save results
ids = test_df_sample['ID']
predprob_one_test = [p[1] for p in predprob_test]
bnp_helper.save_result(ids, predprob_one_test, "xgboost_model_2_result.csv")

print('Done.')